In [1]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("DistilBERT tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)

c:\Users\USER\OneDrive\Desktop\WeReview_DistilBERT\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\USER\OneDrive\Desktop\WeReview_DistilBERT\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as a

DistilBERT tokenizer loaded successfully!
Vocabulary size: 30522


In [2]:
sample_text = "This product is amazing and I really love it!"

tokens = tokenizer(
    sample_text,
    truncation=True,
    max_length=128
)

print("Input text:")
print(sample_text)

print("\nNumber of tokens:", len(tokens["input_ids"]))
print("\nFirst token IDs:")
print(tokens["input_ids"][:20])

Input text:
This product is amazing and I really love it!

Number of tokens: 12

First token IDs:
[101, 2023, 4031, 2003, 6429, 1998, 1045, 2428, 2293, 2009, 999, 102]


In [3]:
import pandas as pd

base_path = "../datasets/salminen"

train_df = pd.read_csv(f"{base_path}/train.csv")
val_df = pd.read_csv(f"{base_path}/validation.csv")
test_df = pd.read_csv(f"{base_path}/test.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nTrain columns:", list(train_df.columns))

Train shape: (32328, 4)
Validation shape: (4042, 4)
Test shape: (4042, 4)

Train columns: ['category', 'rating', 'label', 'text_']


In [4]:
label_map = {
    "OR": 0,
    "CG": 1
}

train_df["label_id"] = train_df["label"].map(label_map)
val_df["label_id"] = val_df["label"].map(label_map)
test_df["label_id"] = test_df["label"].map(label_map)

print("Train labels:")
print(train_df["label_id"].value_counts())

print("\nValidation labels:")
print(val_df["label_id"].value_counts())

print("\nTest labels:")
print(test_df["label_id"].value_counts())

print("\nMissing mapped labels:")
print(
    train_df["label_id"].isna().sum(),
    val_df["label_id"].isna().sum(),
    test_df["label_id"].isna().sum()
)

Train labels:
label_id
0    16171
1    16157
Name: count, dtype: int64

Validation labels:
label_id
0    2022
1    2020
Name: count, dtype: int64

Test labels:
label_id
0    2022
1    2020
Name: count, dtype: int64

Missing mapped labels:
0 0 0


In [5]:
train_texts = train_df["text_"].tolist()
train_labels = train_df["label_id"].tolist()

val_texts = val_df["text_"].tolist()
val_labels = val_df["label_id"].tolist()

test_texts = test_df["text_"].tolist()
test_labels = test_df["label_id"].tolist()

print("Training reviews:", len(train_texts))
print("Validation reviews:", len(val_texts))
print("Test reviews:", len(test_texts))

print("\nExample review:")
print(train_texts[0])

print("\nExample label:")
print(train_labels[0])

Training reviews: 32328
Validation reviews: 4042
Test reviews: 4042

Example review:
I was puzzled to see why this only has four stars, since mine is beautiful and works great. I did have to unscrew it when I first got it because the battery had come loose and was rolling around in the lamp, but I simply stuck it back into place and the lamp works fine. The colors are vibrant and really pop both during the day and when it's lit up at night. And the light is actually a rather bright white, which is emitted from a small stalk of three tiny LED lights gathered into a bunch at the bottom, like grapes. The stake leaves a little something to be desired, though, since it was a little difficult to put together (I had to take the screw out of the pole and then re-insert it, since it was in crooked) and it's not the sturdiest while in the ground. The stake itself is fine enough, but the pole is so narrow that it can't really support the weight of the lantern, which causes the lamp to easily swiv

In [6]:
test_encoding = tokenizer(
    train_texts[0],
    truncation=True,
    max_length=128
)

print("Number of tokens:", len(test_encoding["input_ids"]))
print("Number of attention-mask values:", len(test_encoding["attention_mask"]))
print("First 20 input IDs:", test_encoding["input_ids"][:20])
print("First 20 attention values:", test_encoding["attention_mask"][:20])

Number of tokens: 128
Number of attention-mask values: 128
First 20 input IDs: [101, 1045, 2001, 14909, 2000, 2156, 2339, 2023, 2069, 2038, 2176, 3340, 1010, 2144, 3067, 2003, 3376, 1998, 2573, 2307]
First 20 attention values: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [7]:
batch_encoding = tokenizer(
    train_texts[:4],
    truncation=True,
    max_length=128,
    padding=True
)

print("Number of reviews:", len(batch_encoding["input_ids"]))
print("Shape of input IDs:", len(batch_encoding["input_ids"]), "x", len(batch_encoding["input_ids"][0]))
print("Shape of attention masks:", len(batch_encoding["attention_mask"]), "x", len(batch_encoding["attention_mask"][0]))

Number of reviews: 4
Shape of input IDs: 4 x 128
Shape of attention masks: 4 x 128


In [8]:
import numpy as np

sample_lengths = [
    len(tokenizer(text, truncation=False)["input_ids"])
    for text in train_texts[:1000]
]

print("Average tokens:", round(np.mean(sample_lengths), 2))
print("Median tokens:", int(np.median(sample_lengths)))
print("90th percentile:", int(np.percentile(sample_lengths, 90)))
print("95th percentile:", int(np.percentile(sample_lengths, 95)))
print("Maximum tokens:", max(sample_lengths))

Average tokens: 83.39
Median tokens: 48
90th percentile: 209
95th percentile: 275
Maximum tokens: 469


In [9]:
batch_encoding_256 = tokenizer(
    train_texts[:4],
    truncation=True,
    max_length=256,
    padding=True
)

print("Number of reviews:", len(batch_encoding_256["input_ids"]))
print("Sequence length:", len(batch_encoding_256["input_ids"][0]))
print("Attention mask length:", len(batch_encoding_256["attention_mask"][0]))

Number of reviews: 4
Sequence length: 256
Attention mask length: 256


In [10]:
from transformers import AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("DistilBERT model loaded successfully!")
print("Number of labels:", model.config.num_labels)
print("Parameters:", sum(p.numel() for p in model.parameters()))

c:\Users\USER\OneDrive\Desktop\WeReview_DistilBERT\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 261.53it/s]
[transformers] DistilBertForSequenceClassification LOA

DistilBERT model loaded successfully!
Number of labels: 2
Parameters: 66955010


In [11]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3

print(f"GPU memory allocated: {allocated:.2f} GB")
print(f"GPU memory reserved: {reserved:.2f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 2050
GPU memory allocated: 0.25 GB
GPU memory reserved: 0.29 GB


In [12]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True
)

print("Data collator created successfully!")

Data collator created successfully!


In [13]:
from datasets import Dataset

smoke_texts = train_texts[:16]
smoke_labels = train_labels[:16]

smoke_dataset = Dataset.from_dict({
    "text": smoke_texts,
    "label": smoke_labels
})

print("Smoke-test samples:", len(smoke_dataset))
print(smoke_dataset[0])

Smoke-test samples: 16
{'text': "I was puzzled to see why this only has four stars, since mine is beautiful and works great. I did have to unscrew it when I first got it because the battery had come loose and was rolling around in the lamp, but I simply stuck it back into place and the lamp works fine. The colors are vibrant and really pop both during the day and when it's lit up at night. And the light is actually a rather bright white, which is emitted from a small stalk of three tiny LED lights gathered into a bunch at the bottom, like grapes. The stake leaves a little something to be desired, though, since it was a little difficult to put together (I had to take the screw out of the pole and then re-insert it, since it was in crooked) and it's not the sturdiest while in the ground. The stake itself is fine enough, but the pole is so narrow that it can't really support the weight of the lantern, which causes the lamp to easily swivel around. But once it's in place, it's fine. I have

In [14]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256
    )

smoke_tokenized = smoke_dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenization completed!")
print("Columns:", smoke_tokenized.column_names)
print("Number of samples:", len(smoke_tokenized))

Map: 100%|██████████| 16/16 [00:00<00:00, 454.30 examples/s]

Tokenization completed!
Columns: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']
Number of samples: 16


In [15]:
smoke_tokenized = smoke_tokenized.remove_columns(["text", "token_type_ids"])

smoke_tokenized.set_format("torch")

print("Columns:", smoke_tokenized.column_names)
print("Input IDs shape:", smoke_tokenized[0]["input_ids"].shape)
print("Attention mask shape:", smoke_tokenized[0]["attention_mask"].shape)
print("Label:", smoke_tokenized[0]["label"])

Columns: ['label', 'input_ids', 'attention_mask']
Input IDs shape: torch.Size([256])
Attention mask shape: torch.Size([256])
Label: tensor(0)


In [16]:
from transformers import TrainingArguments

smoke_args = TrainingArguments(
    output_dir="../results/distilbert_smoke_test",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    max_steps=1,
    learning_rate=2e-5,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

print("Smoke-test training configuration created!")
print("Batch size: 1")
print("Max steps: 1")
print("FP16: True")

Smoke-test training configuration created!
Batch size: 1
Max steps: 1
FP16: True


In [17]:
from transformers import Trainer

smoke_trainer = Trainer(
    model=model,
    args=smoke_args,
    train_dataset=smoke_tokenized,
    data_collator=data_collator
)

print("Starting 1-step GPU smoke test...")

smoke_result = smoke_trainer.train()

print("Smoke test completed successfully!")

Starting 1-step GPU smoke test...


Step,Training Loss
1,0.709961


Smoke test completed successfully!


In [18]:
import torch

print("GPU:", torch.cuda.get_device_name(0))

allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3
peak = torch.cuda.max_memory_allocated() / 1024**3

print(f"Current allocated: {allocated:.2f} GB")
print(f"Current reserved: {reserved:.2f} GB")
print(f"Peak allocated: {peak:.2f} GB")

GPU: NVIDIA GeForce RTX 2050
Current allocated: 0.77 GB
Current reserved: 1.10 GB
Peak allocated: 1.02 GB


In [19]:
import gc
import torch

del smoke_trainer
del smoke_result
del smoke_tokenized

gc.collect()
torch.cuda.empty_cache()

print("Smoke-test memory cleared!")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Smoke-test memory cleared!
GPU memory allocated: 0.27 GB
GPU memory reserved: 0.30 GB


In [20]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "text": train_texts,
    "label": train_labels
})

val_dataset = Dataset.from_dict({
    "text": val_texts,
    "label": val_labels
})

test_dataset = Dataset.from_dict({
    "text": test_texts,
    "label": test_labels
})

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 32328
Validation: 4042
Test: 4042


In [21]:
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=32,
    remove_columns=["text"],
    desc="Tokenizing training dataset"
)

print(train_tokenized)

Tokenizing training dataset: 100%|██████████| 32328/32328 [00:05<00:00, 5828.84 examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 32328
})


In [22]:
val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=32,
    remove_columns=["text"],
    desc="Tokenizing validation dataset"
)

print(val_tokenized)

Tokenizing validation dataset: 100%|██████████| 4042/4042 [00:00<00:00, 5099.52 examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4042
})


In [23]:
train_tokenized = train_tokenized.remove_columns(["token_type_ids"])
val_tokenized = val_tokenized.remove_columns(["token_type_ids"])

print("Train columns:", train_tokenized.column_names)
print("Validation columns:", val_tokenized.column_names)

Train columns: ['label', 'input_ids', 'attention_mask']
Validation columns: ['label', 'input_ids', 'attention_mask']


In [24]:
print("Keys:", train_tokenized[0].keys())
print("Label:", train_tokenized[0]["label"])
print("Number of tokens:", len(train_tokenized[0]["input_ids"]))
print("Attention mask length:", len(train_tokenized[0]["attention_mask"]))
print("First 20 token IDs:", train_tokenized[0]["input_ids"][:20])

Keys: dict_keys(['label', 'input_ids', 'attention_mask'])
Label: 0
Number of tokens: 256
Attention mask length: 256
First 20 token IDs: [101, 1045, 2001, 14909, 2000, 2156, 2339, 2023, 2069, 2038, 2176, 3340, 1010, 2144, 3067, 2003, 3376, 1998, 2573, 2307]


In [25]:
model.gradient_checkpointing_enable()

print("Gradient checkpointing enabled:", model.is_gradient_checkpointing)

Gradient checkpointing enabled: True


In [26]:
import gc
import torch
from transformers import TrainingArguments, Trainer

# Clear any previous GPU cache
gc.collect()
torch.cuda.empty_cache()

realistic_args = TrainingArguments(
    output_dir="../results/distilbert_realistic_smoke",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=2,
    learning_rate=2e-5,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=1,
    save_strategy="no",
    report_to="none"
)

realistic_trainer = Trainer(
    model=model,
    args=realistic_args,
    train_dataset=train_tokenized.select(range(16)),
    data_collator=data_collator
)

print("Starting realistic GPU smoke test...")
realistic_result = realistic_trainer.train()

print("\nSmoke test completed!")
print(f"Peak GPU memory allocated: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print(f"Peak GPU memory reserved: {torch.cuda.max_memory_reserved() / 1024**3:.2f} GB")

Starting realistic GPU smoke test...


Step,Training Loss
1,2.843262
2,2.717285



Smoke test completed!
Peak GPU memory allocated: 1.14 GB
Peak GPU memory reserved: 1.26 GB


In [27]:
import gc
import torch

del realistic_trainer
del realistic_result

gc.collect()
torch.cuda.empty_cache()

print("Realistic smoke-test memory cleared!")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Realistic smoke-test memory cleared!
GPU memory allocated: 0.27 GB
GPU memory reserved: 0.31 GB


In [28]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("Evaluation metrics ready!")

Evaluation metrics ready!


In [29]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/distilbert_fake_review",
    
    # Training
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    
    # Validation
    per_device_eval_batch_size=2,
    eval_strategy="epoch",
    
    # Optimization
    learning_rate=2e-5,
    weight_decay=0.01,
    
    # GPU / memory
    fp16=True,
    gradient_checkpointing=True,
    
    # Saving
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    # Logging
    logging_steps=50,
    report_to="none",
    
    # Windows stability
    dataloader_num_workers=0,
    
    # Reproducibility
    seed=42,
)

print("Actual training configuration created successfully!")

Actual training configuration created successfully!


In [30]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer created successfully!")
print(f"Training examples: {len(train_tokenized)}")
print(f"Validation examples: {len(val_tokenized)}")

Trainer created successfully!
Training examples: 32328
Validation examples: 4042


In [31]:
import torch

print("Model device:", next(model.parameters()).device)
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

Model device: cuda:0
GPU: NVIDIA GeForce RTX 2050
GPU memory allocated: 0.27 GB
GPU memory reserved: 0.31 GB


In [32]:
print("🚀 Starting DistilBERT training...")
print("Training: 32,328 reviews")
print("Validation: 4,042 reviews")
print("Epochs: 2")
print("Batch size: 2 | Gradient accumulation: 4")
print("Max length: 256")
print("FP16: ON | Gradient checkpointing: ON")
print("-" * 60)

train_result = trainer.train()

print("\n✅ TRAINING COMPLETED!")
print(train_result)

🚀 Starting DistilBERT training...
Training: 32,328 reviews
Validation: 4,042 reviews
Epochs: 2
Batch size: 2 | Gradient accumulation: 4
Max length: 256
FP16: ON | Gradient checkpointing: ON
------------------------------------------------------------


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.331474,0.125217,0.976497,0.964751,0.989109,0.976778
2,0.223019,0.117565,0.981692,0.971872,0.992079,0.981872


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.48it/s]



✅ TRAINING COMPLETED!
TrainOutput(global_step=8082, training_loss=0.45745865229597427, metrics={'train_runtime': 2948.2342, 'train_samples_per_second': 21.93, 'train_steps_per_second': 2.741, 'total_flos': 1938191568927504.0, 'train_loss': 0.45745865229597427, 'epoch': 2.0})


In [33]:
test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True,
    batch_size=32,
    remove_columns=["text"],
    desc="Tokenizing test dataset"
)

test_tokenized = test_tokenized.remove_columns(["token_type_ids"])

print(test_tokenized)

Tokenizing test dataset: 100%|██████████| 4042/4042 [00:00<00:00, 4828.02 examples/s]

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 4042
})


In [34]:
print("🔒 Running final evaluation on the untouched test set...")

test_output = trainer.predict(test_tokenized)

print("\n========== FINAL TEST RESULTS ==========")
for key, value in test_output.metrics.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

🔒 Running final evaluation on the untouched test set...



========== FINAL TEST RESULTS ==========
test_loss: 0.1144
test_accuracy: 0.9822
test_precision: 0.9710
test_recall: 0.9941
test_f1: 0.9824
test_runtime: 34.8828
test_samples_per_second: 115.8740
test_steps_per_second: 57.9370


In [35]:
import os

final_model_path = "../models/distilbert_fake_review_final"

os.makedirs(final_model_path, exist_ok=True)

# Save the best trained model
trainer.save_model(final_model_path)

# Save the tokenizer
tokenizer.save_pretrained(final_model_path)

print("✅ Model and tokenizer saved successfully!")
print("Saved to:", os.path.abspath(final_model_path))

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]

✅ Model and tokenizer saved successfully!
Saved to: c:\Users\USER\OneDrive\Desktop\WeReview_DistilBERT\models\distilbert_fake_review_final


In [36]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

saved_model_path = "../models/distilbert_fake_review_final"

saved_tokenizer = AutoTokenizer.from_pretrained(saved_model_path)
saved_model = AutoModelForSequenceClassification.from_pretrained(saved_model_path)

print("✅ Saved tokenizer loaded!")
print("✅ Saved model loaded!")
print("Number of labels:", saved_model.config.num_labels)
print("Label mapping:", saved_model.config.id2label)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 248.08it/s]

✅ Saved tokenizer loaded!
✅ Saved model loaded!
Number of labels: 2
Label mapping: {0: 'LABEL_0', 1: 'LABEL_1'}


In [37]:
saved_model.config.id2label = {
    0: "ORIGINAL",
    1: "POTENTIALLY_FAKE"
}

saved_model.config.label2id = {
    "ORIGINAL": 0,
    "POTENTIALLY_FAKE": 1
}

saved_model.save_pretrained(saved_model_path)
saved_tokenizer.save_pretrained(saved_model_path)

print("✅ Label mapping updated and saved!")
print("ID → Label:", saved_model.config.id2label)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.01it/s]

✅ Label mapping updated and saved!
ID → Label: {0: 'ORIGINAL', 1: 'POTENTIALLY_FAKE'}


In [38]:
from transformers import AutoModelForSequenceClassification

check_model = AutoModelForSequenceClassification.from_pretrained(
    saved_model_path
)

print("✅ Reloaded model label mapping:")
print(check_model.config.id2label)
print(check_model.config.label2id)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 427.63it/s]

✅ Reloaded model label mapping:
{0: 'ORIGINAL', 1: 'POTENTIALLY_FAKE'}
{'ORIGINAL': 0, 'POTENTIALLY_FAKE': 1}


In [39]:
import torch

# Use the saved model
check_model = check_model.to("cuda")
check_model.eval()

def predict_review(review_text):
    inputs = saved_tokenizer(
        review_text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    inputs = {key: value.to("cuda") for key, value in inputs.items()}

    with torch.no_grad():
        outputs = check_model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)[0]
    predicted_id = torch.argmax(probabilities).item()

    label = check_model.config.id2label[predicted_id]
    confidence = probabilities[predicted_id].item()

    return label, confidence

# Test review
review = "This product is absolutely amazing! The quality is excellent and I would definitely recommend it."

label, confidence = predict_review(review)

print("Review:", review)
print("Prediction:", label)
print(f"Confidence: {confidence:.2%}")

Review: This product is absolutely amazing! The quality is excellent and I would definitely recommend it.
Prediction: POTENTIALLY_FAKE
Confidence: 99.97%


In [40]:
# Take one untouched test review
sample_idx = 0

real_review = test_df.iloc[sample_idx]["text_"]
actual_label = test_df.iloc[sample_idx]["label"]

predicted_label, confidence = predict_review(real_review)

print("Review:")
print(real_review)

print("\nActual dataset label:", actual_label)
print("Model prediction:", predicted_label)
print(f"Model confidence: {confidence:.2%}")

Review:
Working so far.  If anything changes i will update

Actual dataset label: OR
Model prediction: ORIGINAL
Model confidence: 99.67%


In [41]:
# Find the first CG review in the untouched test set
cg_idx = test_df.index[test_df["label"] == "CG"][0]

cg_review = test_df.loc[cg_idx, "text_"]
actual_cg_label = test_df.loc[cg_idx, "label"]

predicted_cg_label, cg_confidence = predict_review(cg_review)

print("Review:")
print(cg_review)

print("\nActual dataset label:", actual_cg_label)
print("Model prediction:", predicted_cg_label)
print(f"Model confidence: {cg_confidence:.2%}")

Review:
I don't feel like writing a book that focuses on just one person.  I think this book is a great companion to the others, including the first three books in the series.  The book is well written and in good condition.I absolutely loved this book! I was so excited

Actual dataset label: CG
Model prediction: POTENTIALLY_FAKE
Model confidence: 100.00%


In [42]:
correct = 0
total = 10

print("========== BATCH SANITY CHECK ==========\n")

for i in range(total):
    review = test_df.iloc[i]["text_"]
    actual = test_df.iloc[i]["label"]

    predicted, confidence = predict_review(review)

    is_correct = (
        (actual == "OR" and predicted == "ORIGINAL") or
        (actual == "CG" and predicted == "POTENTIALLY_FAKE")
    )

    if is_correct:
        correct += 1

    print(f"Review {i+1}:")
    print(f"Actual    : {actual}")
    print(f"Predicted : {predicted}")
    print(f"Confidence: {confidence:.2%}")
    print(f"Result    : {'✅ Correct' if is_correct else '❌ Incorrect'}")
    print("-" * 50)

print(f"\nBatch result: {correct}/{total} correct")
print(f"Batch accuracy: {correct/total:.2%}")

========== BATCH SANITY CHECK ==========

Review 1:
Actual    : OR
Predicted : ORIGINAL
Confidence: 99.67%
Result    : ✅ Correct
--------------------------------------------------
Review 2:
Actual    : CG
Predicted : POTENTIALLY_FAKE
Confidence: 100.00%
Result    : ✅ Correct
--------------------------------------------------
Review 3:
Actual    : OR
Predicted : POTENTIALLY_FAKE
Confidence: 99.79%
Result    : ❌ Incorrect
--------------------------------------------------
Review 4:
Actual    : OR
Predicted : ORIGINAL
Confidence: 99.99%
Result    : ✅ Correct
--------------------------------------------------
Review 5:
Actual    : OR
Predicted : ORIGINAL
Confidence: 99.96%
Result    : ✅ Correct
--------------------------------------------------
Review 6:
Actual    : CG
Predicted : POTENTIALLY_FAKE
Confidence: 100.00%
Result    : ✅ Correct
--------------------------------------------------
Review 7:
Actual    : CG
Predicted : POTENTIALLY_FAKE
Confidence: 100.00%
Result    : ✅ Correct
------

In [43]:
error_idx = 2

error_review = test_df.iloc[error_idx]["text_"]
error_actual = test_df.iloc[error_idx]["label"]

error_predicted, error_confidence = predict_review(error_review)

print("========== MISCLASSIFIED REVIEW ==========")
print("\nReview:")
print(error_review)

print("\nActual label:", error_actual)
print("Predicted:", error_predicted)
print(f"Confidence: {error_confidence:.2%}")

========== MISCLASSIFIED REVIEW ==========

Review:
Best book on China yet, not just a history of the rulers and dynasties in sequence

Actual label: OR
Predicted: POTENTIALLY_FAKE
Confidence: 99.79%


In [44]:
import pandas as pd

liev_path = "../datasets/lievgarcia/amazon_reviews.txt"

liev_df = pd.read_csv(
    liev_path,
    sep="\t",
    header=None,
    on_bad_lines="skip"
)

print("Shape:", liev_df.shape)
print("\nFirst 5 rows:")
display(liev_df.head())
print("\nColumns:")
print(liev_df.columns.tolist())

Shape: (21001, 9)

First 5 rows:


,0,1,2,3,4,5,6,7,8
0,DOC_ID,LABEL,RATING,VERIFIED_PURCHASE,PRODUCT_CATEGORY,PRODUCT_ID,PRODUCT_TITLE,REVIEW_TITLE,REVIEW_TEXT
1,1,__label1__,4,N,PC,B00008NG7N,"Targus PAUK10U Ultra Mini USB Keypad, Black",useful,"When least you think so, this product will sav..."
2,2,__label1__,4,Y,Wireless,B00LH0Y3NM,Note 3 Battery : Stalion Strength Replacement ...,New era for batteries,Lithium batteries are something new introduced...
3,3,__label1__,3,N,Baby,B000I5UZ1Q,"Fisher-Price Papasan Cradle Swing, Starlight",doesn't swing very well.,I purchased this swing for my baby. She is 6 m...
4,4,__label1__,4,N,Office Products,B003822IRA,Casio MS-80B Standard Function Desktop Calculator,Great computing!,I was looking for an inexpensive desk calcolat...



Columns:
[0, 1, 2, 3, 4, 5, 6, 7, 8]


In [45]:
liev_df = pd.read_csv(
    liev_path,
    sep="\t",
    header=0,
    on_bad_lines="skip"
)

print("Shape:", liev_df.shape)

print("\nColumns:")
print(liev_df.columns.tolist())

print("\nLabel distribution:")
print(liev_df["LABEL"].value_counts())

print("\nUnique labels:")
print(liev_df["LABEL"].unique())

Shape: (21000, 9)

Columns:
['DOC_ID', 'LABEL', 'RATING', 'VERIFIED_PURCHASE', 'PRODUCT_CATEGORY', 'PRODUCT_ID', 'PRODUCT_TITLE', 'REVIEW_TITLE', 'REVIEW_TEXT']

Label distribution:
LABEL
__label1__    10500
__label2__    10500
Name: count, dtype: int64

Unique labels:
<ArrowStringArray>
['__label1__', '__label2__']
Length: 2, dtype: str


In [46]:
import os

print("Files in Lievgarcia folder:")

liev_folder = "../datasets/lievgarcia"

for file in os.listdir(liev_folder):
    print("-", file)

Files in Lievgarcia folder:
- amazon_reviews.txt


In [47]:
print("========== __label1__ EXAMPLES ==========\n")

label1_examples = liev_df[liev_df["LABEL"] == "__label1__"]["REVIEW_TEXT"].head(5)

for i, review in enumerate(label1_examples, 1):
    print(f"{i}. {review}\n")

print("=" * 70)

print("\n========== __label2__ EXAMPLES ==========\n")

label2_examples = liev_df[liev_df["LABEL"] == "__label2__"]["REVIEW_TEXT"].head(5)

for i, review in enumerate(label2_examples, 1):
    print(f"{i}. {review}\n")

========== __label1__ EXAMPLES ==========

1. When least you think so, this product will save the day. Just keep it around just in case you need it for something.

2. Lithium batteries are something new introduced in the market there average developing cost is relatively high but Stallion doesn't compromise on quality and provides us with the best at a low cost.<br />There are so many in built technical assistants that act like a sensor in their particular forté. The battery keeps my phone charged up and it works at every voltage and a high voltage is never risked.

3. I purchased this swing for my baby. She is 6 months now and has pretty much out grown it. It is very loud and doesn't swing very well. It is beautiful though. I love the colors and it has a lot of settings, but I don't think it was worth the money.

4. I was looking for an inexpensive desk calcolatur and here it is. It works and does everything I need. Only issue is that it tilts slightly to one side so when I hit any ke

In [48]:
print("========== VERIFIED PURCHASE ==========")
print(pd.crosstab(
    liev_df["LABEL"],
    liev_df["VERIFIED_PURCHASE"],
    normalize="index"
).round(3))

print("\n========== RATING DISTRIBUTION ==========")
print(pd.crosstab(
    liev_df["LABEL"],
    liev_df["RATING"],
    normalize="index"
).round(3))

print("\n========== PRODUCT CATEGORY ==========")
print(liev_df.groupby("LABEL")["PRODUCT_CATEGORY"].nunique())

========== VERIFIED PURCHASE ==========
VERIFIED_PURCHASE      N      Y
LABEL                          
__label1__         0.726  0.274
__label2__         0.160  0.840

========== RATING DISTRIBUTION ==========
RATING          1      2      3      4      5
LABEL                                        
__label1__  0.085  0.060  0.088  0.190  0.577
__label2__  0.083  0.054  0.090  0.188  0.586

========== PRODUCT CATEGORY ==========
LABEL
__label1__    30
__label2__    30
Name: PRODUCT_CATEGORY, dtype: int64


In [49]:
print("Number of reviews:", len(liev_df))
print("Number of categories:", liev_df["PRODUCT_CATEGORY"].nunique())
print("Label counts:")
print(liev_df["LABEL"].value_counts())

print("\nSample document IDs:")
print(liev_df["DOC_ID"].head(10).tolist())

print("\nCategories:")
print(sorted(liev_df["PRODUCT_CATEGORY"].unique()))

Number of reviews: 21000
Number of categories: 30
Label counts:
LABEL
__label1__    10500
__label2__    10500
Name: count, dtype: int64

Sample document IDs:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Categories:
['Apparel', 'Automotive', 'Baby', 'Beauty', 'Books', 'Camera', 'Electronics', 'Furniture', 'Grocery', 'Health & Personal Care', 'Home', 'Home Entertainment', 'Home Improvement', 'Jewelry', 'Kitchen', 'Lawn and Garden', 'Luggage', 'Musical Instruments', 'Office Products', 'Outdoors', 'PC', 'Pet Products', 'Shoes', 'Sports', 'Tools', 'Toys', 'Video DVD', 'Video Games', 'Watches', 'Wireless']


In [50]:
import torch
from tqdm.auto import tqdm

# Use the already loaded saved model
check_model.eval()

sample_external = liev_df.head(100).copy()

predictions = []
confidences = []

for review in tqdm(
    sample_external["REVIEW_TEXT"],
    desc="Predicting external reviews"
):
    label, confidence = predict_review(review)
    predictions.append(label)
    confidences.append(confidence)

sample_external["model_prediction"] = predictions
sample_external["confidence"] = confidences

print("\n========== EXTERNAL SAMPLE RESULTS ==========")

print("\nDataset label vs model prediction:")
print(
    pd.crosstab(
        sample_external["LABEL"],
        sample_external["model_prediction"]
    )
)

print("\nModel prediction distribution:")
print(sample_external["model_prediction"].value_counts())

print("\nAverage confidence:")
print(sample_external["confidence"].mean())

Predicting external reviews: 100%|██████████| 100/100 [00:01<00:00, 57.70it/s]


========== EXTERNAL SAMPLE RESULTS ==========

Dataset label vs model prediction:
model_prediction  ORIGINAL
LABEL                     
__label1__             100

Model prediction distribution:
model_prediction
ORIGINAL    100
Name: count, dtype: int64

Average confidence:
0.9999054300785065


In [51]:
sample_label2 = liev_df[
    liev_df["LABEL"] == "__label2__"
].head(100).copy()

predictions = []
confidences = []

for review in tqdm(
    sample_label2["REVIEW_TEXT"],
    desc="Predicting __label2__ reviews"
):
    label, confidence = predict_review(review)
    predictions.append(label)
    confidences.append(confidence)

sample_label2["model_prediction"] = predictions
sample_label2["confidence"] = confidences

print("\n========== __label2__ RESULTS ==========")

print("\nModel prediction distribution:")
print(sample_label2["model_prediction"].value_counts())

print("\nAverage confidence:")
print(sample_label2["confidence"].mean())

Predicting __label2__ reviews: 100%|██████████| 100/100 [00:03<00:00, 27.42it/s]


========== __label2__ RESULTS ==========

Model prediction distribution:
model_prediction
ORIGINAL    100
Name: count, dtype: int64

Average confidence:
0.9999712568521499


In [52]:
for label_name in ["__label1__", "__label2__"]:
    subset = liev_df[
        liev_df["LABEL"] == label_name
    ].head(1000)

    predictions = []

    print(f"\nProcessing {label_name}...")

    for review in tqdm(
        subset["REVIEW_TEXT"],
        desc=label_name
    ):
        prediction, _ = predict_review(review)
        predictions.append(prediction)

    print(f"\n{label_name} prediction distribution:")
    print(pd.Series(predictions).value_counts())


Processing __label1__...


__label1__: 100%|██████████| 1000/1000 [00:32<00:00, 30.50it/s]



__label1__ prediction distribution:
ORIGINAL            983
POTENTIALLY_FAKE     17
Name: count, dtype: int64

Processing __label2__...


__label2__: 100%|██████████| 1000/1000 [00:36<00:00, 27.18it/s]


__label2__ prediction distribution:
ORIGINAL            991
POTENTIALLY_FAKE      9
Name: count, dtype: int64


In [53]:
hollenbeck_path = "../datasets/hollenbeck.csv"

hollenbeck_df = pd.read_csv(hollenbeck_path)

print("Shape:", hollenbeck_df.shape)

print("\nColumns:")
print(hollenbeck_df.columns.tolist())

print("\nFirst 5 rows:")
display(hollenbeck_df.head())

print("\nDataset information:")
print(hollenbeck_df.info())

C:\Users\USER\AppData\Local\Temp\ipykernel_21840\3864580836.py:3: DtypeWarning: Columns (16: fake_review_campaign_start_date) have mixed types. Specify dtype option on import or set low_memory=False.
  hollenbeck_df = pd.read_csv(hollenbeck_path)


Shape: (381734, 23)

Columns:
['asin', 'review_id', 'reviewer_id', 'review_title', 'review_text', 'review_rating', 'review_date', 'product_title', 'product_url', 'number_of_helpful', 'number_of_photos', 'photo_thumbnail_urls', 'photo_fullsize_urls', 'asin_url', 'review_url', 'reviewer_url', 'fake_review_campaign_start_date', 'fake_review_product', 'reviewer_classified_fake', 'reviewer_classified_honest', 'reviewer_labeled_fake', 'reviewer_labeled_honest', 'review_is_removed_by_amazon']

First 5 rows:


,asin,review_id,reviewer_id,review_title,review_text,review_rating,review_date,product_title,product_url,number_of_helpful,...,asin_url,review_url,reviewer_url,fake_review_campaign_start_date,fake_review_product,reviewer_classified_fake,reviewer_classified_honest,reviewer_labeled_fake,reviewer_labeled_honest,review_is_removed_by_amazon
0,B003MROHZW,R2RDK8SL1QXN8I,AHAHNBDF4GJ26DAL3VS7MGJDNCKA,Dog muzzle,Serves it purpose.,5,2016-01-22,Downtown Pet Supply Quick Fit Dog Muzzle with ...,https://www.amazon.com/Small-Muzzle-Downtown-P...,0.0,...,https://www.amazon.com/dp/B003MROHZW,https://www.amazon.com/gp/customer-reviews/R2R...,https://www.amazon.com/gp/profile/amzn1.accoun...,2020-02-08,True,True,False,0.0,0.0,0.0
1,B008DZ6JPO,R1CSQAGJT0JXO7,AEZ2DKBYDRPE5O3ENPHV3XRJBMGQ,Awesome product!!,I use this ALL the time!! Looks really nice ...,5,2017-09-25,Winsome 92436 Luggage Rack with Shelf,https://www.amazon.com/Winsome-92436-Luggage-R...,0.0,...,https://www.amazon.com/dp/B008DZ6JPO,https://www.amazon.com/gp/customer-reviews/R1C...,https://www.amazon.com/gp/profile/amzn1.accoun...,NaN,True,True,False,0.0,0.0,0.0
2,B008DZ6JPO,R1XD30XZ3BXO7F,AF52D4YPSCPGLWNYVXZDGG525R2A,Good quality. Color is nice just like the pict...,Well made product. I purchased that refurbishe...,5,2018-11-14,Winsome 92436 Luggage Rack with Shelf,https://www.amazon.com/Winsome-92436-Luggage-R...,0.0,...,https://www.amazon.com/dp/B008DZ6JPO,https://www.amazon.com/gp/customer-reviews/R1X...,https://www.amazon.com/gp/profile/amzn1.accoun...,NaN,True,True,False,0.0,0.0,0.0
3,B008DZ6JPO,R3L61UVBYHB2QK,AFPXQX5HARB76E7GOIUTU32332KQ,Nice Looking and Easy Set Up,"Nice looking, sturdy and easy to set up.",5,2020-02-04,Winsome 92436 Luggage Rack with Shelf,https://www.amazon.com/Winsome-92436-Luggage-R...,0.0,...,https://www.amazon.com/dp/B008DZ6JPO,https://www.amazon.com/gp/customer-reviews/R3L...,https://www.amazon.com/gp/profile/amzn1.accoun...,NaN,True,True,False,0.0,0.0,0.0
4,B00GIHF6AS,R36G7QSYO7QRAS,AFP3AD7L3KWYHHQ4UQAIWJAZX7FQ,Cut pvc like fruit.,", Your browser does not support HTML5 video...",5,2020-03-10,"MAXPOWER Automatic PVC Pipe Cutter, Ratchet-ty...",https://www.amazon.com/MAXPOWER-Cortatubos-aut...,4.0,...,https://www.amazon.com/dp/B00GIHF6AS,https://www.amazon.com/gp/customer-reviews/R36...,https://www.amazon.com/gp/profile/amzn1.accoun...,2020-02-14,True,True,False,1.0,0.0,0.0



Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 381734 entries, 0 to 381733
Data columns (total 23 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   asin                             381734 non-null  str    
 1   review_id                        381734 non-null  str    
 2   reviewer_id                      381734 non-null  str    
 3   review_title                     380808 non-null  str    
 4   review_text                      381734 non-null  str    
 5   review_rating                    381734 non-null  int64  
 6   review_date                      381734 non-null  str    
 7   product_title                    267611 non-null  str    
 8   product_url                      267611 non-null  str    
 9   number_of_helpful                376413 non-null  float64
 10  number_of_photos                 376413 non-null  float64
 11  photo_thumbnail_urls             22911 non-null   str 

In [54]:
label_columns = [
    "reviewer_classified_fake",
    "reviewer_classified_honest",
    "reviewer_labeled_fake",
    "reviewer_labeled_honest",
    "review_is_removed_by_amazon"
]

for col in label_columns:
    print("\n" + "=" * 60)
    print(col)
    print(hollenbeck_df[col].value_counts(dropna=False))


reviewer_classified_fake
reviewer_classified_fake
False    324067
True      57667
Name: count, dtype: int64

reviewer_classified_honest
reviewer_classified_honest
False    359120
True      22614
Name: count, dtype: int64

reviewer_labeled_fake
reviewer_labeled_fake
NaN    301453
0.0     58143
1.0     22138
Name: count, dtype: int64

reviewer_labeled_honest
reviewer_labeled_honest
NaN    301453
0.0     69479
1.0     10802
Name: count, dtype: int64

review_is_removed_by_amazon
review_is_removed_by_amazon
0.0    256614
NaN     86175
1.0     38945
Name: count, dtype: int64


In [55]:
label_columns = [
    "reviewer_classified_fake",
    "reviewer_classified_honest",
    "reviewer_labeled_fake",
    "reviewer_labeled_honest",
    "review_is_removed_by_amazon"
]

for col in label_columns:
    print("\n" + "=" * 60)
    print(col)
    print(hollenbeck_df[col].value_counts(dropna=False))


reviewer_classified_fake
reviewer_classified_fake
False    324067
True      57667
Name: count, dtype: int64

reviewer_classified_honest
reviewer_classified_honest
False    359120
True      22614
Name: count, dtype: int64

reviewer_labeled_fake
reviewer_labeled_fake
NaN    301453
0.0     58143
1.0     22138
Name: count, dtype: int64

reviewer_labeled_honest
reviewer_labeled_honest
NaN    301453
0.0     69479
1.0     10802
Name: count, dtype: int64

review_is_removed_by_amazon
review_is_removed_by_amazon
0.0    256614
NaN     86175
1.0     38945
Name: count, dtype: int64


In [56]:
print("Rows marked classified as BOTH fake and honest:")
print(
    hollenbeck_df[
        (hollenbeck_df["reviewer_classified_fake"] == True) &
        (hollenbeck_df["reviewer_classified_honest"] == True)
    ].shape[0]
)

print("\nRows marked classified as NEITHER:")
print(
    hollenbeck_df[
        (hollenbeck_df["reviewer_classified_fake"] == False) &
        (hollenbeck_df["reviewer_classified_honest"] == False)
    ].shape[0]
)

print("\nRows with BOTH labeled_fake=1 and labeled_honest=1:")
print(
    hollenbeck_df[
        (hollenbeck_df["reviewer_labeled_fake"] == 1) &
        (hollenbeck_df["reviewer_labeled_honest"] == 1)
    ].shape[0]
)

print("\nRows with labeled_fake=1 OR labeled_honest=1:")
print(
    (
        (hollenbeck_df["reviewer_labeled_fake"] == 1) |
        (hollenbeck_df["reviewer_labeled_honest"] == 1)
    ).sum()
)

Rows marked classified as BOTH fake and honest:
0

Rows marked classified as NEITHER:
301453

Rows with BOTH labeled_fake=1 and labeled_honest=1:
0

Rows with labeled_fake=1 OR labeled_honest=1:
32940


In [57]:
print("Label combinations:")
print(
    hollenbeck_df[
        hollenbeck_df["reviewer_labeled_fake"].notna() |
        hollenbeck_df["reviewer_labeled_honest"].notna()
    ][
        ["reviewer_labeled_fake", "reviewer_labeled_honest"]
    ].value_counts(dropna=False)
)

Label combinations:
reviewer_labeled_fake  reviewer_labeled_honest
0.0                    0.0                        47341
1.0                    0.0                        22138
0.0                    1.0                        10802
Name: count, dtype: int64


In [58]:
# Create explicitly labeled Hollenbeck evaluation set

hollenbeck_eval = hollenbeck_df[
    (hollenbeck_df["reviewer_labeled_fake"] == 1) |
    (hollenbeck_df["reviewer_labeled_honest"] == 1)
].copy()

# Create binary labels:
# 0 = ORIGINAL/HONEST
# 1 = POTENTIALLY_FAKE/FAKE
hollenbeck_eval["label"] = (
    hollenbeck_eval["reviewer_labeled_fake"] == 1
).astype(int)

print("External evaluation dataset shape:", hollenbeck_eval.shape)

print("\nLabel distribution:")
print(hollenbeck_eval["label"].value_counts())

print("\nReview text missing:", hollenbeck_eval["review_text"].isna().sum())

print("\nFirst 3 samples:")
print(
    hollenbeck_eval[
        ["review_text", "review_rating", "label"]
    ].head(3).to_string(index=False)
)

External evaluation dataset shape: (32940, 24)

Label distribution:
label
1    22138
0    10802
Name: count, dtype: int64

Review text missing: 0

First 3 samples:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             review_text  review_rating  label
  ,  Your browser does not support HTML5 video.,        ,   This pipe cutter is super convenient because it uses ratchet mechanism that allow you to continue to apply amount of forces with very minimum effort. The blade will be pushed down more and more until the pipe is fully

In [59]:
# Check Hollenbeck review length

hollenbeck_eval["text_length"] = (
    hollenbeck_eval["review_text"]
    .astype(str)
    .str.split()
    .str.len()
)

print("Text length statistics:")
print(hollenbeck_eval["text_length"].describe())

print("\nPercentiles:")
print(
    hollenbeck_eval["text_length"].quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
    )
)

Text length statistics:
count    32940.000000
mean        53.348452
std         55.156820
min          1.000000
25%         21.000000
50%         41.000000
75%         68.000000
max       2501.000000
Name: text_length, dtype: float64

Percentiles:
0.50      41.0
0.75      68.0
0.90     108.0
0.95     142.0
0.99     246.0
1.00    2501.0
Name: text_length, dtype: float64


In [60]:
# Check how many Hollenbeck reviews exceed the model's 256-token limit

sample_texts = hollenbeck_eval["review_text"].astype(str).tolist()

token_lengths = []

for text in sample_texts:
    tokens = tokenizer(
        text,
        truncation=False,
        add_special_tokens=True
    )["input_ids"]
    token_lengths.append(len(tokens))

hollenbeck_eval["token_length"] = token_lengths

print("Token length statistics:")
print(hollenbeck_eval["token_length"].describe())

print("\nToken percentiles:")
print(
    hollenbeck_eval["token_length"].quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
    )
)

print(
    "\nReviews exceeding 256 tokens:",
    (hollenbeck_eval["token_length"] > 256).sum()
)

print(
    "Percentage exceeding 256 tokens:",
    (hollenbeck_eval["token_length"] > 256).mean() * 100
)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (627 > 512). Running this sequence through the model will result in indexing errors


Token length statistics:
count    32940.000000
mean        66.713631
std         69.037214
min          3.000000
25%         28.000000
50%         51.000000
75%         85.000000
max       3511.000000
Name: token_length, dtype: float64

Token percentiles:
0.50      51.0
0.75      85.0
0.90     132.0
0.95     175.0
0.99     308.0
1.00    3511.0
Name: token_length, dtype: float64

Reviews exceeding 256 tokens: 571
Percentage exceeding 256 tokens: 1.7334547662416515


In [61]:
from datasets import Dataset

# Keep only the columns we need
hollenbeck_model_df = hollenbeck_eval[
    ["review_text", "label"]
].copy()

# Convert pandas DataFrame to Hugging Face Dataset
hollenbeck_dataset = Dataset.from_pandas(
    hollenbeck_model_df,
    preserve_index=False
)

# Tokenize using the SAME tokenizer and max_length as training
def tokenize_hollenbeck(batch):
    return tokenizer(
        batch["review_text"],
        truncation=True,
        max_length=256
    )

hollenbeck_tokenized = hollenbeck_dataset.map(
    tokenize_hollenbeck,
    batched=True,
    remove_columns=["review_text"]
)

print(hollenbeck_tokenized)

print("\nColumns:")
print(hollenbeck_tokenized.column_names)

print("\nFirst sample:")
print(hollenbeck_tokenized[0])

Map: 100%|██████████| 32940/32940 [00:07<00:00, 4522.42 examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 32940
})

Columns:
['label', 'input_ids', 'token_type_ids', 'attention_mask']

First sample:
{'label': 1, 'input_ids': [101, 1010, 2115, 16602, 2515, 2025, 2490, 16129, 2629, 2678, 1012, 1010, 1010, 2023, 8667, 16343, 2003, 3565, 14057, 2138, 2009, 3594, 9350, 20318, 7337, 2008, 3499, 2017, 2000, 3613, 2000, 6611, 3815, 1997, 2749, 2007, 2200, 6263, 3947, 1012, 1996, 6085, 2097, 2022, 3724, 2091, 2062, 1998, 2062, 2127, 1996, 8667, 2003, 3929, 3013, 1012, 1996, 8667, 2097, 2022, 1037, 3442, 3013, 1998, 2097, 2022, 6179, 1999, 2116, 5097, 1012, 2023, 8667, 16343, 2003, 2081, 2007, 13061, 2061, 2009, 3685, 2022, 18399, 1012, 1045, 2066, 2009, 2138, 2009, 16142, 2026, 2398, 6669, 1998, 1045, 2123, 1521, 1056, 2342, 2000, 6611, 2205, 2172, 2486, 2000, 3013, 4317, 26189, 6169, 1012, 2307, 4031, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [62]:
hollenbeck_tokenized = hollenbeck_tokenized.remove_columns(
    ["token_type_ids"]
)

print(hollenbeck_tokenized)

print("\nColumns:")
print(hollenbeck_tokenized.column_names)

Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 32940
})

Columns:
['label', 'input_ids', 'attention_mask']


In [65]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

# Same padding strategy used during training
external_data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True
)

external_args = TrainingArguments(
    output_dir="../results/hollenbeck_eval",
    per_device_eval_batch_size=4,
    fp16=True,
    report_to="none",
    dataloader_num_workers=0
)

external_trainer = Trainer(
    model=model,
    args=external_args,
    data_collator=external_data_collator
)

print("Starting external evaluation...")

hollenbeck_results = external_trainer.predict(
    hollenbeck_tokenized
)

print("\nExternal evaluation completed.")
print(hollenbeck_results.metrics)

Starting external evaluation...



External evaluation completed.
{'test_loss': 7.1261773109436035, 'test_model_preparation_time': 0.0, 'test_runtime': 209.8966, 'test_samples_per_second': 156.934, 'test_steps_per_second': 39.234}


In [66]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import numpy as np

# Get predictions
predictions = np.argmax(
    hollenbeck_results.predictions,
    axis=1
)

# True labels
true_labels = np.array(
    hollenbeck_tokenized["label"]
)

# Calculate metrics
accuracy = accuracy_score(true_labels, predictions)
precision = precision_score(true_labels, predictions, zero_division=0)
recall = recall_score(true_labels, predictions, zero_division=0)
f1 = f1_score(true_labels, predictions, zero_division=0)

print("=" * 60)
print("HOLLENBECK EXTERNAL EVALUATION")
print("=" * 60)

print(f"Accuracy : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision * 100:.2f}%)")
print(f"Recall   : {recall:.4f} ({recall * 100:.2f}%)")
print(f"F1 Score : {f1:.4f} ({f1 * 100:.2f}%)")

print("\nConfusion Matrix:")
print(confusion_matrix(true_labels, predictions))

print("\nClassification Report:")
print(
    classification_report(
        true_labels,
        predictions,
        target_names=["ORIGINAL/HONEST", "POTENTIALLY_FAKE"],
        digits=4
    )
)

HOLLENBECK EXTERNAL EVALUATION
Accuracy : 0.3136 (31.36%)
Precision: 0.3743 (37.43%)
Recall   : 0.0317 (3.17%)
F1 Score : 0.0584 (5.84%)

Confusion Matrix:
[[ 9630  1172]
 [21437   701]]

Classification Report:
                  precision    recall  f1-score   support

 ORIGINAL/HONEST     0.3100    0.8915    0.4600     10802
POTENTIALLY_FAKE     0.3743    0.0317    0.0584     22138

        accuracy                         0.3136     32940
       macro avg     0.3421    0.4616    0.2592     32940
    weighted avg     0.3532    0.3136    0.1901     32940



In [67]:
# Examine how the explicitly labeled reviews relate to other fake-review indicators

labeled = hollenbeck_df[
    (hollenbeck_df["reviewer_labeled_fake"] == 1) |
    (hollenbeck_df["reviewer_labeled_honest"] == 1)
].copy()

print("Total explicitly labeled:", len(labeled))

print("\n--- Labeled fake reviews ---")
fake_labeled = labeled[labeled["reviewer_labeled_fake"] == 1]

print("Count:", len(fake_labeled))
print(
    "Also classified fake:",
    fake_labeled["reviewer_classified_fake"].sum()
)
print(
    "Also classified honest:",
    fake_labeled["reviewer_classified_honest"].sum()
)
print(
    "Removed by Amazon:",
    (fake_labeled["review_is_removed_by_amazon"] == 1).sum()
)

print("\n--- Labeled honest reviews ---")
honest_labeled = labeled[labeled["reviewer_labeled_honest"] == 1]

print("Count:", len(honest_labeled))
print(
    "Also classified fake:",
    honest_labeled["reviewer_classified_fake"].sum()
)
print(
    "Also classified honest:",
    honest_labeled["reviewer_classified_honest"].sum()
)
print(
    "Removed by Amazon:",
    (honest_labeled["review_is_removed_by_amazon"] == 1).sum()
)

Total explicitly labeled: 32940

--- Labeled fake reviews ---
Count: 22138
Also classified fake: 22043
Also classified honest: 95
Removed by Amazon: 9274

--- Labeled honest reviews ---
Count: 10802
Also classified fake: 74
Also classified honest: 10728
Removed by Amazon: 355


In [68]:
# Compare basic characteristics of Hollenbeck fake vs honest labeled reviews

fake_group = hollenbeck_eval[hollenbeck_eval["label"] == 1]
honest_group = hollenbeck_eval[hollenbeck_eval["label"] == 0]

print("=" * 60)
print("FAKE-LABELED REVIEWS")
print("=" * 60)

print("Count:", len(fake_group))
print("Average rating:", fake_group["review_rating"].mean())
print("Average word count:", fake_group["text_length"].mean())
print("Average token count:", fake_group["token_length"].mean())

print("\nRating distribution:")
print(fake_group["review_rating"].value_counts().sort_index())

print("\n" + "=" * 60)
print("HONEST-LABELED REVIEWS")
print("=" * 60)

print("Count:", len(honest_group))
print("Average rating:", honest_group["review_rating"].mean())
print("Average word count:", honest_group["text_length"].mean())
print("Average token count:", honest_group["token_length"].mean())

print("\nRating distribution:")
print(honest_group["review_rating"].value_counts().sort_index())

FAKE-LABELED REVIEWS
Count: 22138
Average rating: 4.821257566175806
Average word count: 59.70232179962056
Average token count: 74.12047158731593

Rating distribution:
review_rating
1      547
2      187
3      229
4      750
5    20425
Name: count, dtype: int64

HONEST-LABELED REVIEWS
Count: 10802
Average rating: 4.349564895389743
Average word count: 40.326606184039996
Average token count: 51.53379003888169

Rating distribution:
review_rating
1     836
2     431
3     623
4    1143
5    7769
Name: count, dtype: int64


In [69]:
# Identify Hollenbeck fake-labeled reviews that DistilBERT predicted as ORIGINAL

hollenbeck_results_df = hollenbeck_eval.copy()

hollenbeck_results_df["prediction"] = predictions

false_negatives = hollenbeck_results_df[
    (hollenbeck_results_df["label"] == 1) &
    (hollenbeck_results_df["prediction"] == 0)
].copy()

print("False negatives:", len(false_negatives))

print("\nFirst 10 false negatives:")
print(
    false_negatives[
        ["review_rating", "text_length", "token_length", "review_text"]
    ].head(10).to_string(index=False)
)

False negatives: 21437

First 10 false negatives:
 review_rating  text_length  token_length                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     review_text
             5           99           114                                                                                                                   

In [70]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("=" * 70)
print("HOLLENBECK PERFORMANCE BY STAR RATING")
print("=" * 70)

for rating in sorted(hollenbeck_results_df["review_rating"].unique()):
    
    subset = hollenbeck_results_df[
        hollenbeck_results_df["review_rating"] == rating
    ]
    
    y_true = subset["label"]
    y_pred = subset["prediction"]
    
    print(f"\nRating: {rating} stars")
    print(f"Samples : {len(subset)}")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall: {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1: {f1_score(y_true, y_pred, zero_division=0):.4f}")

HOLLENBECK PERFORMANCE BY STAR RATING

Rating: 1 stars
Samples : 1383
Accuracy: 0.6052
Precision: 0.5047
Recall: 0.0987
F1: 0.1651

Rating: 2 stars
Samples : 618
Accuracy: 0.6796
Precision: 0.3103
Recall: 0.0481
F1: 0.0833

Rating: 3 stars
Samples : 852
Accuracy: 0.6925
Precision: 0.2381
Recall: 0.0655
F1: 0.1027

Rating: 4 stars
Samples : 1893
Accuracy: 0.5621
Precision: 0.2074
Recall: 0.0373
F1: 0.0633

Rating: 5 stars
Samples : 28194
Accuracy: 0.2632
Precision: 0.3866
Recall: 0.0291
F1: 0.0542


In [71]:
print("=" * 70)
print("SALMINEN TRAINING DATA — RATING DISTRIBUTION")
print("=" * 70)

print("\nOverall:")
print(
    salminen_clean["rating"]
    .value_counts()
    .sort_index()
)

print("\nBy label:")
print(
    pd.crosstab(
        salminen_clean["label"],
        salminen_clean["rating"],
        normalize="index"
    ).round(4)
)

print("\nCounts by label and rating:")
print(
    pd.crosstab(
        salminen_clean["label"],
        salminen_clean["rating"]
    )
)

SALMINEN TRAINING DATA — RATING DISTRIBUTION

Overall:


NameError: name 'salminen_clean' is not defined

In [72]:
# Load the cleaned Salminen dataset
salminen_clean = pd.read_csv(
    "../datasets/salminen/fake_reviews_clean.csv"
)

print("=" * 70)
print("SALMINEN TRAINING DATA — RATING DISTRIBUTION")
print("=" * 70)

print("\nOverall:")
print(
    salminen_clean["rating"]
    .value_counts()
    .sort_index()
)

print("\nBy label:")
print(
    pd.crosstab(
        salminen_clean["label"],
        salminen_clean["rating"],
        normalize="index"
    ).round(4)
)

print("\nCounts by label and rating:")
print(
    pd.crosstab(
        salminen_clean["label"],
        salminen_clean["rating"]
    )
)

SALMINEN TRAINING DATA — RATING DISTRIBUTION

Overall:
rating
1.0     2155
2.0     1967
3.0     3785
4.0     7959
5.0    24546
Name: count, dtype: int64

By label:
rating     1.0     2.0     3.0     4.0     5.0
label                                         
CG      0.0526  0.0476  0.0966  0.1938  0.6093
OR      0.0540  0.0497  0.0907  0.2001  0.6054

Counts by label and rating:
rating   1.0   2.0   3.0   4.0    5.0
label                                
CG      1063   962  1951  3914  12307
OR      1092  1005  1834  4045  12239


In [73]:
salminen_clean["text_length"] = (
    salminen_clean["text_"]
    .astype(str)
    .str.split()
    .str.len()
)

print("=" * 70)
print("SALMINEN TEXT LENGTH BY LABEL")
print("=" * 70)

print(
    salminen_clean.groupby("label")["text_length"]
    .agg(["count", "mean", "median", "min", "max"])
)

print("\nPercentiles by label:")

for label in ["OR", "CG"]:
    values = salminen_clean[
        salminen_clean["label"] == label
    ]["text_length"]

    print(f"\n{label}:")
    print(
        values.quantile(
            [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
        )
    )

SALMINEN TEXT LENGTH BY LABEL
       count       mean  median  min  max
label                                    
CG     20197  61.317077    35.0    1  318
OR     20215  73.643136    42.0    5  373

Percentiles by label:

OR:
0.25     23.0
0.50     42.0
0.75     93.0
0.90    190.0
0.95    261.0
0.99    329.0
Name: text_length, dtype: float64

CG:
0.25     19.0
0.50     35.0
0.75     77.0
0.90    156.0
0.95    211.2
0.99    270.0
Name: text_length, dtype: float64


In [74]:
from pathlib import Path

hollenbeck_path = Path("../datasets/hollenbeck.csv")

if hollenbeck_path.exists():
    hollenbeck_path.unlink()
    print("Hollenbeck dataset deleted permanently from the project.")
else:
    print("Hollenbeck dataset not found at:", hollenbeck_path)

print("\nRemaining datasets:")
for path in Path("../datasets").rglob("*"):
    if path.is_file():
        print(path)

Hollenbeck dataset deleted permanently from the project.

Remaining datasets:
..\datasets\lievgarcia\amazon_reviews.txt
..\datasets\salminen\fake reviews dataset.csv
..\datasets\salminen\fake_reviews_clean.csv
..\datasets\salminen\test.csv
..\datasets\salminen\train.csv
..\datasets\salminen\validation.csv


In [75]:
lievgarcia_path = "../datasets/lievgarcia/amazon_reviews.txt"

lievgarcia_df = pd.read_csv(
    lievgarcia_path,
    sep="\t",
    header=0
)

print("Shape:", lievgarcia_df.shape)

print("\nColumns:")
print(lievgarcia_df.columns.tolist())

print("\nLabel distribution:")
print(lievgarcia_df["LABEL"].value_counts())

print("\nFirst 5 rows:")
print(
    lievgarcia_df[
        ["LABEL", "RATING", "REVIEW_TITLE", "REVIEW_TEXT"]
    ].head().to_string(index=False)
)

Shape: (21000, 9)

Columns:
['DOC_ID', 'LABEL', 'RATING', 'VERIFIED_PURCHASE', 'PRODUCT_CATEGORY', 'PRODUCT_ID', 'PRODUCT_TITLE', 'REVIEW_TITLE', 'REVIEW_TEXT']

Label distribution:
LABEL
__label1__    10500
__label2__    10500
Name: count, dtype: int64

First 5 rows:
     LABEL  RATING             REVIEW_TITLE                                                                                                                                                                                                                                                                                                                                                                                                          REVIEW_TEXT
__label1__       4                   useful                                                                                                                                                                                                                                               

In [76]:
with open(lievgarcia_path, "r", encoding="utf-8") as f:
    lines = [next(f) for _ in range(5)]

print("First 5 raw lines:")
for line in lines:
    print(line.rstrip())

print("\nLast 5 raw lines:")
with open(lievgarcia_path, "r", encoding="utf-8") as f:
    all_lines = f.readlines()

for line in all_lines[-5:]:
    print(line.rstrip())

First 5 raw lines:
DOC_ID	LABEL	RATING	VERIFIED_PURCHASE	PRODUCT_CATEGORY	PRODUCT_ID	PRODUCT_TITLE	REVIEW_TITLE	REVIEW_TEXT
1	__label1__	4	N	PC	B00008NG7N	"Targus PAUK10U Ultra Mini USB Keypad, Black"	useful	"When least you think so, this product will save the day. Just keep it around just in case you need it for something."
2	__label1__	4	Y	Wireless	B00LH0Y3NM	Note 3 Battery : Stalion Strength Replacement 3200mAh Li-Ion Battery for Samsung Galaxy Note 3 [24-Month Warranty] with NFC Chip + Google Wallet Capable	New era for batteries	Lithium batteries are something new introduced in the market there average developing cost is relatively high but Stallion doesn't compromise on quality and provides us with the best at a low cost.<br />There are so many in built technical assistants that act like a sensor in their particular forté. The battery keeps my phone charged up and it works at every voltage and a high voltage is never risked.
3	__label1__	3	N	Baby	B000I5UZ1Q	"Fisher-Price Papasan C

In [77]:
from pathlib import Path

lievgarcia_folder = Path("../datasets/lievgarcia")

print("Files in Lievgarcia folder:")
for path in lievgarcia_folder.iterdir():
    print(path.name)

Files in Lievgarcia folder:
amazon_reviews.txt


In [78]:
# Create the same label convention used by our DistilBERT model

lievgarcia_df["label"] = (
    lievgarcia_df["LABEL"] == "__label1__"
).astype(int)

print("Label mapping:")
print("__label1__ → 1 → POTENTIALLY_FAKE")
print("__label2__ → 0 → ORIGINAL")

print("\nFinal label distribution:")
print(lievgarcia_df["label"].value_counts().sort_index())

print("\nLabel + original label check:")
print(
    lievgarcia_df[["LABEL", "label"]]
    .value_counts()
    .sort_index()
)

Label mapping:
__label1__ → 1 → POTENTIALLY_FAKE
__label2__ → 0 → ORIGINAL

Final label distribution:
label
0    10500
1    10500
Name: count, dtype: int64

Label + original label check:
LABEL       label
__label1__  1        10500
__label2__  0        10500
Name: count, dtype: int64


In [79]:
# Create the same label convention used by our DistilBERT model

lievgarcia_df["label"] = (
    lievgarcia_df["LABEL"] == "__label1__"
).astype(int)

print("Label mapping:")
print("__label1__ → 1 → POTENTIALLY_FAKE")
print("__label2__ → 0 → ORIGINAL")

print("\nFinal label distribution:")
print(lievgarcia_df["label"].value_counts().sort_index())

print("\nLabel + original label check:")
print(
    lievgarcia_df[["LABEL", "label"]]
    .value_counts()
    .sort_index()
)

Label mapping:
__label1__ → 1 → POTENTIALLY_FAKE
__label2__ → 0 → ORIGINAL

Final label distribution:
label
0    10500
1    10500
Name: count, dtype: int64

Label + original label check:
LABEL       label
__label1__  1        10500
__label2__  0        10500
Name: count, dtype: int64


In [80]:
# Create the same label convention used by our DistilBERT model
# 0 = ORIGINAL
# 1 = POTENTIALLY_FAKE

lievgarcia_df["label"] = (
    lievgarcia_df["LABEL"] == "__label1__"
).astype(int)

print("Label mapping:")
print("__label1__ → 1 → POTENTIALLY_FAKE")
print("__label2__ → 0 → ORIGINAL")

print("\nFinal label distribution:")
print(lievgarcia_df["label"].value_counts().sort_index())

print("\nLabel + original label check:")
print(
    lievgarcia_df[["LABEL", "label"]]
    .value_counts()
    .sort_index()
)


Label mapping:
__label1__ → 1 → POTENTIALLY_FAKE
__label2__ → 0 → ORIGINAL

Final label distribution:
label
0    10500
1    10500
Name: count, dtype: int64

Label + original label check:
LABEL       label
__label1__  1        10500
__label2__  0        10500
Name: count, dtype: int64


In [81]:
# Check review text quality

print("Missing review text:")
print(lievgarcia_df["REVIEW_TEXT"].isna().sum())

print("\nEmpty / whitespace-only reviews:")
print(
    (lievgarcia_df["REVIEW_TEXT"].fillna("").str.strip() == "").sum()
)

print("\nDuplicate review texts:")
print(lievgarcia_df["REVIEW_TEXT"].duplicated().sum())

print("\nReview length statistics:")
print(lievgarcia_df["REVIEW_TEXT"].str.len().describe())

Missing review text:
0

Empty / whitespace-only reviews:
0

Duplicate review texts:
0

Review length statistics:
count    21000.000000
mean       372.301714
std        480.509498
min         98.000000
25%        156.000000
50%        236.000000
75%        388.000000
max      16188.000000
Name: REVIEW_TEXT, dtype: float64


In [82]:
# Create a clean text-only dataframe

lievgarcia_clean = lievgarcia_df[["REVIEW_TEXT", "label"]].copy()

# Rename columns to match our training pipeline
lievgarcia_clean = lievgarcia_clean.rename(
    columns={
        "REVIEW_TEXT": "text"
    }
)

# Remove accidental leading/trailing whitespace
lievgarcia_clean["text"] = lievgarcia_clean["text"].str.strip()

print("Clean dataset shape:", lievgarcia_clean.shape)

print("\nColumns:")
print(lievgarcia_clean.columns.tolist())

print("\nLabel distribution:")
print(lievgarcia_clean["label"].value_counts().sort_index())

print("\nFirst 3 reviews:")
print(lievgarcia_clean.head(3))

Clean dataset shape: (21000, 2)

Columns:
['text', 'label']

Label distribution:
label
0    10500
1    10500
Name: count, dtype: int64

First 3 reviews:
                                                text  label
0  When least you think so, this product will sav...      1
1  Lithium batteries are something new introduced...      1
2  I purchased this swing for my baby. She is 6 m...      1


In [83]:
from sklearn.model_selection import train_test_split

# First: 80% training, 20% temporary
liev_train, liev_temp = train_test_split(
    lievgarcia_clean,
    test_size=0.20,
    stratify=lievgarcia_clean["label"],
    random_state=42
)

# Second: split the temporary 20% equally
# → 10% validation, 10% test
liev_val, liev_test = train_test_split(
    liev_temp,
    test_size=0.50,
    stratify=liev_temp["label"],
    random_state=42
)

print("Dataset sizes:")
print("Train      :", len(liev_train))
print("Validation :", len(liev_val))
print("Test       :", len(liev_test))

print("\nTrain label distribution:")
print(liev_train["label"].value_counts().sort_index())

print("\nValidation label distribution:")
print(liev_val["label"].value_counts().sort_index())

print("\nTest label distribution:")
print(liev_test["label"].value_counts().sort_index())

# Check for text overlap between splits
train_texts = set(liev_train["text"])
val_texts = set(liev_val["text"])
test_texts = set(liev_test["text"])

print("\nText overlap:")
print("Train ∩ Validation:", len(train_texts & val_texts))
print("Train ∩ Test      :", len(train_texts & test_texts))
print("Validation ∩ Test :", len(val_texts & test_texts))

Dataset sizes:
Train      : 16800
Validation : 2100
Test       : 2100

Train label distribution:
label
0    8400
1    8400
Name: count, dtype: int64

Validation label distribution:
label
0    1050
1    1050
Name: count, dtype: int64

Test label distribution:
label
0    1050
1    1050
Name: count, dtype: int64

Text overlap:
Train ∩ Validation: 0
Train ∩ Test      : 0
Validation ∩ Test : 0


In [84]:
from transformers import AutoTokenizer
import numpy as np

# Load the same tokenizer used by our DistilBERT model
liev_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenize without truncation to measure the actual lengths
sample_texts = liev_train["text"].tolist()

token_lengths = []

for text in sample_texts:
    tokens = liev_tokenizer(
        text,
        truncation=False,
        add_special_tokens=True
    )["input_ids"]

    token_lengths.append(len(tokens))

token_lengths = np.array(token_lengths)

print("Number of training reviews:", len(token_lengths))

print("\nToken length statistics:")
print("Mean   :", round(token_lengths.mean(), 2))
print("Median :", int(np.median(token_lengths)))
print("90th   :", int(np.percentile(token_lengths, 90)))
print("95th   :", int(np.percentile(token_lengths, 95)))
print("99th   :", int(np.percentile(token_lengths, 99)))
print("Max    :", int(token_lengths.max()))

print("\nReviews exceeding different lengths:")
print("> 128 :", int((token_lengths > 128).sum()))
print("> 256 :", int((token_lengths > 256).sum()))
print("> 384 :", int((token_lengths > 384).sum()))
print("> 512 :", int((token_lengths > 512).sum()))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (940 > 512). Running this sequence through the model will result in indexing errors


Number of training reviews: 16800

Token length statistics:
Mean   : 89.58
Median : 56
90th   : 170
95th   : 260
99th   : 576
Max    : 2120

Reviews exceeding different lengths:
> 128 : 2611
> 256 : 862
> 384 : 416
> 512 : 222


In [85]:
from pathlib import Path

# Create a folder for the Lievgarcia processed dataset
liev_processed_dir = Path("../datasets/lievgarcia/processed")
liev_processed_dir.mkdir(parents=True, exist_ok=True)

# Save train / validation / test splits
liev_train.to_csv(liev_processed_dir / "train.csv", index=False)
liev_val.to_csv(liev_processed_dir / "validation.csv", index=False)
liev_test.to_csv(liev_processed_dir / "test.csv", index=False)

print("Saved files:")

for file in sorted(liev_processed_dir.glob("*.csv")):
    print(f"{file.name}: {file.stat().st_size / 1024:.1f} KB")

Saved files:
test.csv: 781.4 KB
train.csv: 6172.4 KB
validation.csv: 793.1 KB


In [86]:
from datasets import Dataset

MAX_LENGTH = 256

# Convert training DataFrame to Hugging Face Dataset
liev_train_ds = Dataset.from_pandas(
    liev_train[["text", "label"]],
    preserve_index=False
)

# Tokenization function
def tokenize_lievgarcia(batch):
    return liev_tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

# Tokenize training data
liev_train_tokenized = liev_train_ds.map(
    tokenize_lievgarcia,
    batched=True,
    batch_size=1000,
    desc="Tokenizing training data"
)

# Remove text because the model only needs tokens + labels
liev_train_tokenized = liev_train_tokenized.remove_columns(["text"])

print("Tokenized training dataset:")
print(liev_train_tokenized)

print("\nColumns:")
print(liev_train_tokenized.column_names)

print("\nNumber of training samples:")
print(len(liev_train_tokenized))

Tokenizing training data: 100%|██████████| 16800/16800 [00:04<00:00, 3706.77 examples/s]

Tokenized training dataset:
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 16800
})

Columns:
['label', 'input_ids', 'token_type_ids', 'attention_mask']

Number of training samples:
16800


In [87]:
# DistilBERT does not use token_type_ids
if "token_type_ids" in liev_train_tokenized.column_names:
    liev_train_tokenized = liev_train_tokenized.remove_columns(
        ["token_type_ids"]
    )

print("Final training columns:")
print(liev_train_tokenized.column_names)

print("\nTraining samples:")
print(len(liev_train_tokenized))

print("\nFirst sample:")
print(liev_train_tokenized[0])

Final training columns:
['label', 'input_ids', 'attention_mask']

Training samples:
16800

First sample:
{'label': 1, 'input_ids': [101, 2023, 2003, 1037, 2307, 8934, 2000, 2224, 2000, 6366, 2115, 2304, 13038, 2006, 2115, 4451, 1012, 2009, 2573, 6669, 1010, 1998, 2009, 2003, 2054, 2027, 2109, 2000, 6366, 2115, 2304, 13038, 12145, 999, 2009, 2003, 4276, 1996, 4965, 1010, 1998, 1045, 2572, 2200, 8510, 2007, 2026, 5309, 1010, 1998, 2026, 4451, 2003, 2085, 3154, 1997, 2304, 13038, 2044, 8192, 1997, 2023, 6366, 2099, 8934, 1024, 1007, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [88]:
# Convert validation DataFrame to Hugging Face Dataset
liev_val_ds = Dataset.from_pandas(
    liev_val[["text", "label"]],
    preserve_index=False
)

# Tokenize validation data
liev_val_tokenized = liev_val_ds.map(
    tokenize_lievgarcia,
    batched=True,
    batch_size=1000,
    desc="Tokenizing validation data"
)

# Remove unnecessary columns
columns_to_remove = ["text"]

if "token_type_ids" in liev_val_tokenized.column_names:
    columns_to_remove.append("token_type_ids")

liev_val_tokenized = liev_val_tokenized.remove_columns(columns_to_remove)

print("Tokenized validation dataset:")
print(liev_val_tokenized)

print("\nColumns:")
print(liev_val_tokenized.column_names)

print("\nNumber of validation samples:")
print(len(liev_val_tokenized))

Tokenizing validation data: 100%|██████████| 2100/2100 [00:00<00:00, 3449.41 examples/s]

Tokenized validation dataset:
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 2100
})

Columns:
['label', 'input_ids', 'attention_mask']

Number of validation samples:
2100


In [89]:
# Convert test DataFrame to Hugging Face Dataset
liev_test_ds = Dataset.from_pandas(
    liev_test[["text", "label"]],
    preserve_index=False
)

# Tokenize test data
liev_test_tokenized = liev_test_ds.map(
    tokenize_lievgarcia,
    batched=True,
    batch_size=1000,
    desc="Tokenizing test data"
)

# Remove unnecessary columns
columns_to_remove = ["text"]

if "token_type_ids" in liev_test_tokenized.column_names:
    columns_to_remove.append("token_type_ids")

liev_test_tokenized = liev_test_tokenized.remove_columns(columns_to_remove)

print("Tokenized test dataset:")
print(liev_test_tokenized)

print("\nColumns:")
print(liev_test_tokenized.column_names)

print("\nNumber of test samples:")
print(len(liev_test_tokenized))

Tokenizing test data: 100%|██████████| 2100/2100 [00:00<00:00, 3635.62 examples/s]

Tokenized test dataset:
Dataset({
    features: ['label', 'input_ids', 'attention_mask'],
    num_rows: 2100
})

Columns:
['label', 'input_ids', 'attention_mask']

Number of test samples:
2100


In [90]:
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# Clear GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())

# Fresh DistilBERT model
smoke_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label={
        0: "ORIGINAL",
        1: "POTENTIALLY_FAKE"
    },
    label2id={
        "ORIGINAL": 0,
        "POTENTIALLY_FAKE": 1
    }
)

smoke_model = smoke_model.to("cuda")

# Enable gradient checkpointing
smoke_model.gradient_checkpointing_enable()

smoke_args = TrainingArguments(
    output_dir="../models/lievgarcia_smoke_test",
    max_steps=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=1,
    report_to="none",
    dataloader_num_workers=0,
    seed=42
)

smoke_trainer = Trainer(
    model=smoke_model,
    args=smoke_args,
    train_dataset=liev_train_tokenized
)

print("\nStarting 2-step smoke test...")

smoke_trainer.train()

print("\nSmoke test completed successfully.")

if torch.cuda.is_available():
    print(
        f"Peak GPU allocated: "
        f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
    )
    print(
        f"Peak GPU reserved: "
        f"{torch.cuda.max_memory_reserved() / 1024**3:.2f} GB"
    )

# Free memory
del smoke_trainer
del smoke_model

torch.cuda.empty_cache()

print("\nGPU memory cleared.")

GPU: NVIDIA GeForce RTX 2050
CUDA available: True


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 350.83it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Starting 2-step smoke test...


Step,Training Loss
1,2.803467
2,2.653809


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.56s/it]



Smoke test completed successfully.
Peak GPU allocated: 2.14 GB
Peak GPU reserved: 2.33 GB

GPU memory cleared.


In [91]:
import torch
from transformers import AutoModelForSequenceClassification

# Clear GPU memory first
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Create a fresh DistilBERT model
liev_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2,
    id2label={
        0: "ORIGINAL",
        1: "POTENTIALLY_FAKE"
    },
    label2id={
        "ORIGINAL": 0,
        "POTENTIALLY_FAKE": 1
    }
)

# Move model to GPU
liev_model = liev_model.to("cuda")

# Enable gradient checkpointing
liev_model.gradient_checkpointing_enable()

print("Model loaded successfully.")
print("Device:", next(liev_model.parameters()).device)
print("Parameters:", sum(p.numel() for p in liev_model.parameters()))

print("\nLabel mapping:")
print(liev_model.config.id2label)
print(liev_model.config.label2id)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 182.87it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully.
Device: cuda:0
Parameters: 66955010

Label mapping:
{0: 'ORIGINAL', 1: 'POTENTIALLY_FAKE'}
{'ORIGINAL': 0, 'POTENTIALLY_FAKE': 1}


In [92]:
from transformers import TrainingArguments

liev_training_args = TrainingArguments(
    output_dir="../models/distilbert_lievgarcia_fake_review",

    # Training
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    # Validation
    per_device_eval_batch_size=2,
    eval_strategy="epoch",

    # Optimization
    learning_rate=2e-5,
    weight_decay=0.01,

    # GPU memory
    fp16=True,
    gradient_checkpointing=True,

    # Saving
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,

    # Best model selection
    metric_for_best_model="f1",
    greater_is_better=True,

    # Logging
    logging_steps=50,
    report_to="none",

    # Reproducibility / Windows stability
    dataloader_num_workers=0,
    seed=42
)

print("Training configuration created successfully.")
print(liev_training_args)

Training configuration created successfully.
TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=

In [93]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_liev_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        pos_label=1,
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("Evaluation metrics function created successfully.")

Evaluation metrics function created successfully.


In [94]:
from transformers import Trainer

liev_trainer = Trainer(
    model=liev_model,
    args=liev_training_args,
    train_dataset=liev_train_tokenized,
    eval_dataset=liev_val_tokenized,
    compute_metrics=compute_liev_metrics
)

print("Trainer created successfully.")

print("\nTraining samples:", len(liev_train_tokenized))
print("Validation samples:", len(liev_val_tokenized))
print("Epochs:", liev_training_args.num_train_epochs)
print("Batch size:", liev_training_args.per_device_train_batch_size)
print("Gradient accumulation:", liev_training_args.gradient_accumulation_steps)
print("Max length:", MAX_LENGTH)

Trainer created successfully.

Training samples: 16800
Validation samples: 2100
Epochs: 2
Batch size: 2
Gradient accumulation: 4
Max length: 256


In [95]:
print("Starting Lievgarcia DistilBERT training...")
print("Training samples:", len(liev_train_tokenized))
print("Validation samples:", len(liev_val_tokenized))
print("Epochs:", liev_training_args.num_train_epochs)
print("Expected training steps:", 4200)

liev_train_result = liev_trainer.train()

print("\n" + "=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)

print("\nTraining metrics:")
print(liev_train_result.metrics)

Starting Lievgarcia DistilBERT training...
Training samples: 16800
Validation samples: 2100
Epochs: 2
Expected training steps: 4200


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,2.305405,0.577829,0.692857,0.687327,0.707619,0.697325
2,2.007948,0.610946,0.700952,0.684764,0.744762,0.713504


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.26it/s]



TRAINING COMPLETED

Training metrics:
{'train_runtime': 2686.0348, 'train_samples_per_second': 12.509, 'train_steps_per_second': 1.564, 'total_flos': 2225452297420800.0, 'train_loss': 2.211681958153134, 'epoch': 2.0}


In [96]:
print("Evaluating on the untouched Lievgarcia test set...")

liev_test_results = liev_trainer.evaluate(
    eval_dataset=liev_test_tokenized
)

print("\n" + "=" * 60)
print("FINAL LIEVGARCIA TEST RESULTS")
print("=" * 60)

for key, value in liev_test_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

Evaluating on the untouched Lievgarcia test set...


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
2.007948,0.610641,2,0.699524,0.689937,0.724762,0.706921



FINAL LIEVGARCIA TEST RESULTS
eval_loss: 0.6106
eval_accuracy: 0.6995
eval_precision: 0.6899
eval_recall: 0.7248
eval_f1: 0.7069


In [97]:
# Analyze metadata relationships with the Lievgarcia labels

print("=" * 60)
print("VERIFIED PURCHASE BY LABEL")
print("=" * 60)

print(
    pd.crosstab(
        lievgarcia_df["LABEL"],
        lievgarcia_df["VERIFIED_PURCHASE"],
        normalize="index"
    ).round(4)
)

print("\n" + "=" * 60)
print("RATING BY LABEL")
print("=" * 60)

print(
    pd.crosstab(
        lievgarcia_df["LABEL"],
        lievgarcia_df["RATING"],
        normalize="index"
    ).round(4)
)

print("\n" + "=" * 60)
print("PRODUCT CATEGORY COUNT")
print("=" * 60)

print(
    lievgarcia_df["PRODUCT_CATEGORY"].nunique()
)

print("\nCategories per label:")
print(
    lievgarcia_df.groupby("LABEL")["PRODUCT_CATEGORY"]
    .nunique()
)

VERIFIED PURCHASE BY LABEL
VERIFIED_PURCHASE       N       Y
LABEL                            
__label1__         0.7260  0.2740
__label2__         0.1599  0.8401

RATING BY LABEL
RATING           1       2       3       4       5
LABEL                                             
__label1__  0.0847  0.0597  0.0882  0.1904  0.5770
__label2__  0.0827  0.0538  0.0897  0.1880  0.5858

PRODUCT CATEGORY COUNT
30

Categories per label:
LABEL
__label1__    30
__label2__    30
Name: PRODUCT_CATEGORY, dtype: int64


In [98]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier

# Use only VERIFIED_PURCHASE from the training split
X_train_meta = (
    liev_train[["VERIFIED_PURCHASE"]]
    .replace({"N": 0, "Y": 1})
)

y_train_meta = liev_train["label"]

# Small decision tree to test the predictive signal
meta_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

meta_model.fit(X_train_meta, y_train_meta)

train_meta_pred = meta_model.predict(X_train_meta)

print("Training-only metadata baseline:")
print("Accuracy :", round(accuracy_score(y_train_meta, train_meta_pred), 4))
print("Precision:", round(precision_score(y_train_meta, train_meta_pred), 4))
print("Recall   :", round(recall_score(y_train_meta, train_meta_pred), 4))
print("F1       :", round(f1_score(y_train_meta, train_meta_pred), 4))

print("\nDecision tree rule:")
print("VERIFIED_PURCHASE =", meta_model.tree_.threshold[0])

KeyError: "None of [Index(['VERIFIED_PURCHASE'], dtype='str')] are in the [columns]"

In [99]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeClassifier

# Recover the original row indices from the train split
train_indices = liev_train.index

# Get VERIFIED_PURCHASE from the original dataset
X_train_meta = (
    lievgarcia_df.loc[train_indices, ["VERIFIED_PURCHASE"]]
    .replace({"N": 0, "Y": 1})
)

y_train_meta = liev_train["label"]

# Small decision tree to measure the signal
meta_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

meta_model.fit(X_train_meta, y_train_meta)

train_meta_pred = meta_model.predict(X_train_meta)

print("Training-only VERIFIED_PURCHASE baseline:")
print("Accuracy :", round(accuracy_score(y_train_meta, train_meta_pred), 4))
print("Precision:", round(precision_score(y_train_meta, train_meta_pred), 4))
print("Recall   :", round(recall_score(y_train_meta, train_meta_pred), 4))
print("F1       :", round(f1_score(y_train_meta, train_meta_pred), 4))

print("\nFeature values:")
print(X_train_meta["VERIFIED_PURCHASE"].value_counts().sort_index())

Training-only VERIFIED_PURCHASE baseline:
Accuracy : 0.7823
Precision: 0.8175
Recall   : 0.7269
F1       : 0.7696

Feature values:
VERIFIED_PURCHASE
0    7469
1    9331
Name: count, dtype: int64


In [100]:
print("Missing review titles:")
print(lievgarcia_df["REVIEW_TITLE"].isna().sum())

print("\nEmpty review titles:")
print(
    (lievgarcia_df["REVIEW_TITLE"].fillna("").str.strip() == "").sum()
)

print("\nAverage title length:")
print(
    lievgarcia_df["REVIEW_TITLE"]
    .fillna("")
    .str.len()
    .mean()
)

print("\nExample titles:")
print(
    lievgarcia_df[["LABEL", "REVIEW_TITLE", "REVIEW_TEXT"]]
    .head(5)
    .to_string(index=False)
)

Missing review titles:
0

Empty review titles:
0

Average title length:
25.97747619047619

Example titles:
     LABEL             REVIEW_TITLE                                                                                                                                                                                                                                                                                                                                                                                                          REVIEW_TEXT
__label1__                   useful                                                                                                                                                                                                                                                                                                 When least you think so, this product will save the day. Just keep it around just in case you need it for something.
__label1__ 

In [101]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Combine review title + review text
train_combined = (
    lievgarcia_df.loc[liev_train.index, "REVIEW_TITLE"].astype(str)
    + " "
    + lievgarcia_df.loc[liev_train.index, "REVIEW_TEXT"].astype(str)
)

val_combined = (
    lievgarcia_df.loc[liev_val.index, "REVIEW_TITLE"].astype(str)
    + " "
    + lievgarcia_df.loc[liev_val.index, "REVIEW_TEXT"].astype(str)
)

# TF-IDF
tfidf_liev = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf_liev.fit_transform(train_combined)
X_val_tfidf = tfidf_liev.transform(val_combined)

y_train = liev_train["label"]
y_val = liev_val["label"]

print("TF-IDF shapes:")
print("Train:", X_train_tfidf.shape)
print("Validation:", X_val_tfidf.shape)

# Fast linear classifier
tfidf_model = LogisticRegression(
    max_iter=1000,
    C=2.0,
    random_state=42
)

tfidf_model.fit(X_train_tfidf, y_train)

val_pred = tfidf_model.predict(X_val_tfidf)

print("\nTitle + Text TF-IDF validation results:")
print("Accuracy :", round(accuracy_score(y_val, val_pred), 4))
print("Precision:", round(precision_score(y_val, val_pred), 4))
print("Recall   :", round(recall_score(y_val, val_pred), 4))
print("F1       :", round(f1_score(y_val, val_pred), 4))

TF-IDF shapes:
Train: (16800, 20000)
Validation: (2100, 20000)

Title + Text TF-IDF validation results:
Accuracy : 0.6838
Precision: 0.6881
Recall   : 0.6724
F1       : 0.6802


In [102]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Recover metadata using the SAME train/validation indices
metadata_cols = [
    "VERIFIED_PURCHASE",
    "RATING",
    "PRODUCT_CATEGORY"
]

X_train_meta = lievgarcia_df.loc[
    liev_train.index, metadata_cols
].copy()

X_val_meta = lievgarcia_df.loc[
    liev_val.index, metadata_cols
].copy()

y_train_meta = liev_train["label"]
y_val_meta = liev_val["label"]

# Treat all metadata as categorical
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            metadata_cols
        )
    ]
)

metadata_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            C=1.0,
            random_state=42
        )
    )
])

metadata_model.fit(X_train_meta, y_train_meta)

val_meta_pred = metadata_model.predict(X_val_meta)

print("Metadata-only validation results:")
print("Accuracy :", round(accuracy_score(y_val_meta, val_meta_pred), 4))
print("Precision:", round(precision_score(y_val_meta, val_meta_pred), 4))
print("Recall   :", round(recall_score(y_val_meta, val_meta_pred), 4))
print("F1       :", round(f1_score(y_val_meta, val_meta_pred), 4))

Metadata-only validation results:
Accuracy : 0.7871
Precision: 0.8339
Recall   : 0.7171
F1       : 0.7711


In [103]:
print("=" * 60)
print("UNIQUE PRODUCTS")
print("=" * 60)

print("Total reviews:", len(lievgarcia_df))
print("Unique PRODUCT_IDs:", lievgarcia_df["PRODUCT_ID"].nunique())

print("\nAverage reviews per product:")
print(
    round(
        len(lievgarcia_df) / lievgarcia_df["PRODUCT_ID"].nunique(),
        2
    )
)

print("\n" + "=" * 60)
print("LABEL DISTRIBUTION BY PRODUCT")
print("=" * 60)

product_label_counts = (
    lievgarcia_df
    .groupby("PRODUCT_ID")["LABEL"]
    .nunique()
)

print("Products containing only one label:")
print((product_label_counts == 1).sum())

print("Products containing both labels:")
print((product_label_counts == 2).sum())

print("\n" + "=" * 60)
print("LABEL DISTRIBUTION BY PRODUCT CATEGORY")
print("=" * 60)

category_label_counts = (
    lievgarcia_df
    .groupby("PRODUCT_CATEGORY")["LABEL"]
    .nunique()
)

print("Categories containing only one label:")
print((category_label_counts == 1).sum())

print("Categories containing both labels:")
print((category_label_counts == 2).sum())

UNIQUE PRODUCTS
Total reviews: 21000
Unique PRODUCT_IDs: 18857

Average reviews per product:
1.11

LABEL DISTRIBUTION BY PRODUCT
Products containing only one label:
18414
Products containing both labels:
443

LABEL DISTRIBUTION BY PRODUCT CATEGORY
Categories containing only one label:
0
Categories containing both labels:
30
